In [8]:
import jax
import jax.numpy as jnp
import optax
from tqdm import tqdm
import numpy as np

In [9]:
import sys
sys.path.append("../")
from pqcqec.circuits.generate import generate_random_circuit
from pqcqec.circuits.modify import tokenize_qiskit_circuit

from pqcqec.models.pqc_models import StateInputModelInterleavedPQCModel
from pqcqec.noise.simple_noise import PennylaneNoisyGates
from pqcqec.simulate.simulate import get_input_data, run_circuit_with_noise_model

from pqcqec.training.jax_loss_functions import jax_pure_state_fidelity, jax_mse_complex_loss

from pqcqec.utils.jax_utils import JAXStateDataset, JAXDataLoader

In [10]:
seed=0
num_qubits=3
num_gates=4
gate_blocks=4
num_data=2500
num_test=100
batch_size=10
epochs=5

In [11]:

# Set random seed for reproducibility
jax_prng_keys = jax.random.split(jax.random.PRNGKey(seed), 3).flatten() # Split gives us (3,2) shape, flatten to (6,) 
print(f"Using Seed and JAX PRNG Keys: {seed, jax_prng_keys}")

# Generate ideal data
ideal_train_data = get_input_data(num_qubits, num_data, seed=jax_prng_keys[0])

# Generate noise
# train_noise = JAXNoise(x_rad=jnp.pi/100, z_rad=jnp.pi/100, shape=(num_data, num_gates * 2), seed=jax_prng_keys[1])
# print(noise_dist)

noise_model = PennylaneNoisyGates(x_rad=jnp.pi/10, 
                                  z_rad=0, 
                                  delta_x=0, 
                                  delta_z=0, seed=jax_prng_keys[1])

# Create dataset and dataloader
gate_dist = {"h": 0.5, "cx": 0.25, "x": 0.25}

train_dataset = JAXStateDataset(ideal_train_data)
train_dataloader = JAXDataLoader(train_dataset, batch_size=batch_size, shuffle=True, seed=jax_prng_keys[2])

# Generate random circuit list
qiskit_random_circuit = generate_random_circuit(
    num_qubits=num_qubits,
    num_gates=num_gates,
    gate_dist=gate_dist,
    seed=seed
)


qiskit_uncomp_circuit = qiskit_random_circuit

uncomp_circuit_ops = tokenize_qiskit_circuit(qiskit_uncomp_circuit)

print(f"Uncomputation Circuit Ops: {uncomp_circuit_ops}")

Using Seed and JAX PRNG Keys: (0, Array([1797259609, 2579123966,  928981903, 3453687069, 4146024105,
       2718843009], dtype=uint32))
Uncomputation Circuit Ops: [('x', [2], []), ('x', [1], []), ('h', [1], []), ('h', [1], [])]


In [12]:
def train_pqc_model_no_uncomp(model, dataloader, optimizer, schedule, main_loss_fn=jax_mse_complex_loss, epochs=1):

    no_noise_model = PennylaneNoisyGates(x_rad=0, z_rad=0, delta_x=0, delta_z=0, seed=0)
        
    @jax.jit
    def update_step(params, opt_state, ideal_data):
        """Perform a single update step for the model parameters."""
        
        def loss_fn(p):
            measured = model(ideal_data, params=p)
            simulated = run_circuit_with_noise_model(model.circuit_ops, ideal_data, no_noise_model, model.num_qubits)
            return main_loss_fn(simulated, measured)

        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)

        # Fidelity after parameter update
        measured = model(ideal_data, params=new_params)
        simulated = run_circuit_with_noise_model(model.circuit_ops, ideal_data, no_noise_model, model.num_qubits)

        fidelity = jax_pure_state_fidelity(simulated, measured)

        return opt_state, new_params, loss, fidelity

    for e in range(epochs):
        print(f"Epoch {e + 1}/{epochs}")
        # Reset op state for each epoch    
        opt_state = optimizer.init(model.pqc_params)
        data_iterator = tqdm(dataloader, desc="Training", total=len(dataloader), leave=False, unit='batch')
        
        # Initialize lists to track metrics for this epoch
        epoch_fidelities = []
        epoch_losses = []

        for i, batch in enumerate(data_iterator):

            # ideal_data = batch  # Assuming the first element is the ideal data
            # print(f'Batch Shape: {batch}')
            ideal_data = batch[0]  # Assuming the first element is the ideal data
            # print(f'Ideal Data Shape: {ideal_data.shape}')
            # print(f'Ideal Data \n: {ideal_data}')

            opt_state, params, loss, fidelity = update_step(model.pqc_params, opt_state, ideal_data)
            model.pqc_params = params
            
            # Track metrics
            epoch_fidelities.append(float(fidelity))
            epoch_losses.append(float(loss))

            current_lr = schedule(i)

            data_iterator.set_postfix_str(f"Fidelity (Ideal, Measured): {fidelity:.4e}, Loss: {loss:.4e}, LR: {current_lr:.4e}")
        
        # Print mean metrics at the end of each epoch
        mean_fidelity = np.mean(epoch_fidelities)
        mean_loss = np.mean(epoch_losses)
        print(f"Epoch {e+1} summary - Mean Fidelity: {mean_fidelity:.4e}, Mean Loss: {mean_loss:.4e}")


In [13]:
# Initialize model
model = StateInputModelInterleavedPQCModel(circuit_ops=uncomp_circuit_ops,
                                        num_qubits=num_qubits,
                                        noise_model=noise_model,
                                        pqc_blocks=1,
                                        gate_blocks=gate_blocks,
                                        seed=jax_prng_keys[4])

print(f"Model Parameters Shape: {model.pqc_params.shape}")
print(f"Model Parameter Count: {model.pqc_params.size}")

# Define optimizer
TOTAL_STEPS = int(num_data / batch_size)
WARMUP_STEPS = int(0.1 * TOTAL_STEPS)
RESTART_PERIOD = int(0.25 * TOTAL_STEPS)

INIT_LR = 1e-5
PEAK_LR = 1e-2
MIN_LR = 5e-5

# 1. Warmup schedule
warmup = optax.linear_schedule(
    init_value=INIT_LR,
    end_value=PEAK_LR,
    transition_steps=WARMUP_STEPS
)

# 2. Cosine decay with restarts
def cosine_with_restart_schedule(step):
    step_in_period = step % RESTART_PERIOD
    cosine = 0.5 * (1 + jnp.cos(jnp.pi * step_in_period / RESTART_PERIOD))
    return MIN_LR + (PEAK_LR - MIN_LR) * cosine

# 3. Stitch warmup + cosine
schedule = optax.join_schedules(
    schedules=[warmup, cosine_with_restart_schedule],
    boundaries=[WARMUP_STEPS]
)

# 4. Optimizer chain
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.scale_by_adam(eps=1e-8),
    optax.add_decayed_weights(weight_decay=1e-4),
    optax.scale_by_schedule(schedule),
    optax.scale(-1.0)
)


train_pqc_model_no_uncomp(model, train_dataloader, optimizer, schedule, epochs=epochs)

# Test the model

# Generate test data
ideal_test_data = get_input_data(num_qubits, num_test, seed=jax_prng_keys[5])

print(f'Ideal Test Data Shape: {ideal_test_data.shape}')
print(f'Running circuit with noise model on test data...')
noisy_state = run_circuit_with_noise_model(
    uncomp_circuit_ops,
    ideal_test_data,
    noise_model,
    num_qubits,
    batched=True,
)

no_noise_model = PennylaneNoisyGates(x_rad=0, z_rad=0, delta_x=0, delta_z=0, seed=0)

simulated = run_circuit_with_noise_model(
    uncomp_circuit_ops,
    ideal_test_data,
    no_noise_model,
    num_qubits,
    batched=True,
)

print(f'Running PQC model on test data...')
pqc_state = model.run_model_batch(ideal_test_data)
batched_fidelity = jax.vmap(jax_pure_state_fidelity, in_axes=(0, 0))    

fidelity_ideal_noisy = batched_fidelity(simulated, noisy_state)
fidelity_ideal_pqc = batched_fidelity(simulated, pqc_state)

print(f"Fidelity (Ideal, Noisy): {jnp.mean(fidelity_ideal_noisy):.4e}")
print(f"Fidelity (Ideal, PQC): {jnp.mean(fidelity_ideal_pqc):.4e}")


Model Parameters Shape: (1, 3, 3)
Model Parameter Count: 9
Epoch 1/5


Epoch 1 summary - Mean Fidelity: 1.4425e-01, Mean Loss: 1.8979e-01
Epoch 2/5


Epoch 2 summary - Mean Fidelity: 9.1166e-01, Mean Loss: 1.1805e-02
Epoch 3/5


Epoch 3 summary - Mean Fidelity: 9.9945e-01, Mean Loss: 7.2744e-05
Epoch 4/5


Epoch 4 summary - Mean Fidelity: 1.0000e+00, Mean Loss: 5.1410e-08
Epoch 5/5


Epoch 5 summary - Mean Fidelity: 1.0000e+00, Mean Loss: 7.1949e-08
Ideal Test Data Shape: (100, 8)
Running circuit with noise model on test data...
Running PQC model on test data...
Fidelity (Ideal, Noisy): 8.7423e-01
Fidelity (Ideal, PQC): 1.0000e+00
